# PoC tuần 1 — Mô hình có sẵn thấy biển trên ảnh Mapillary được bao nhiêu?

1. Runtime → Change runtime type → **T4 GPU**.
2. Chạy ô 1, chọn file `poc-data.zip` (tạo bởi `node tools/poc/fetch-signs.mjs`).
3. Chạy hết. Cuối cùng tải về `poc-results.zip`.

Mô hình: **YOLO-World** (phát hiện theo câu chữ, không cần train) với lớp `traffic sign`. Một ảnh được tính là **thấy biển** khi có hộp phát hiện nằm đúng hướng mà biển phải xuất hiện (tính từ vị trí, hướng camera và tiêu cự), lệch tối đa `TOL_DEG`.

In [ ]:
# 1. Tải dữ liệu lên
!pip -q install ultralytics
import zipfile, os, io
from google.colab import files
up = files.upload()
# Zip tạo trên Windows có thể dùng dấu gạch ngược (chr 92) trong đường dẫn: đổi sang / khi giải nén.
with zipfile.ZipFile(io.BytesIO(next(iter(up.values())))) as z:
    for m in z.infolist():
        name = m.filename.replace(chr(92), '/')
        if name.endswith('/'): continue
        dst = os.path.join('data', *name.split('/'))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with open(dst, 'wb') as f: f.write(z.read(m))
print(sorted(os.listdir('data')), len(os.listdir('data/images')), 'ảnh')

In [ ]:
# 2. Chạy YOLO-World trên toàn bộ ảnh
import math, pandas as pd, numpy as np
from PIL import Image
from ultralytics import YOLOWorld

PROMPTS = ['traffic sign']
CONF_MIN, TOL_DEG, IMGSZ = 0.05, 12, 1280

imgs = pd.read_csv('data/images.csv', encoding='utf-8-sig')
imgs['pano'] = imgs['pano'].astype(str).str.lower() == 'true'
imgs['mapillary_saw'] = imgs['mapillary_saw'].astype(str).str.lower() == 'true'
imgs = imgs[imgs.image_id.astype(str).map(lambda i: os.path.exists(f'data/images/{i}.jpg'))].reset_index(drop=True)

model = YOLOWorld('yolov8x-worldv2.pt')
model.set_classes(PROMPTS)

def det_angle(cx, w, h, row):
    # Góc (độ) của tâm hộp so với hướng camera; dương = bên phải.
    if row.pano:
        return (cx / w - 0.5) * 360
    f = row.focal if row.focal == row.focal and row.focal else 0.8  # Mapillary: tiêu cự chuẩn hóa theo cạnh dài
    return math.degrees(math.atan((cx - w / 2) / (f * max(w, h))))

dets = []
for i in range(0, len(imgs), 16):
    batch = imgs.iloc[i:i + 16]
    res = model.predict([f'data/images/{r}.jpg' for r in batch.image_id], conf=CONF_MIN, imgsz=IMGSZ, verbose=False)
    for row, r in zip(batch.itertuples(), res):
        h, w = r.orig_shape
        for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
            a = det_angle((x1 + x2) / 2, w, h, row)
            err = abs((a - row.off_deg + 540) % 360 - 180)
            dets.append(dict(image_id=row.image_id, conf=c, ang_err=err, x1=x1, y1=y1, x2=x2, y2=y2, box_px=max(x2 - x1, y2 - y1)))
dets = pd.DataFrame(dets)
print(len(imgs), 'ảnh,', len(dets), 'hộp phát hiện')

In [ ]:
# 3. Tỉ lệ thấy biển
ok = dets[dets.ang_err <= TOL_DEG].sort_values('conf', ascending=False).drop_duplicates('image_id')
imgs = imgs.merge(ok[['image_id', 'conf', 'box_px', 'x1', 'y1', 'x2', 'y2']], on='image_id', how='left')
imgs['dist_bin'] = pd.cut(imgs.dist_m, [5, 15, 25, 35], labels=['5–15 m', '15–25 m', '25–35 m'])

def rate(df, t):
    return round(100 * (df.conf >= t).mean())

rows = []
for t in [0.10, 0.25]:
    rows.append(dict(nhom='Tất cả', conf=t, so_anh=len(imgs), thay_pct=rate(imgs, t)))
    for k, g in imgs.groupby('dist_bin', observed=True):
        rows.append(dict(nhom=f'Khoảng cách {k}', conf=t, so_anh=len(g), thay_pct=rate(g, t)))
    for k, g in imgs.groupby('pano'):
        rows.append(dict(nhom='Ảnh 360°' if k else 'Ảnh thường', conf=t, so_anh=len(g), thay_pct=rate(g, t)))
    for k, g in imgs.groupby('mapillary_saw'):
        rows.append(dict(nhom='Mapillary cũng thấy' if k else 'Mapillary không thấy', conf=t, so_anh=len(g), thay_pct=rate(g, t)))
summary = pd.DataFrame(rows)
display(summary)

per_sign = imgs.assign(thay=imgs.conf >= 0.25).groupby(['sign_id', 'sign_type']).agg(
    so_anh=('image_id', 'size'), so_nguoi_chup=('creator', 'nunique'), thay_pct=('thay', lambda s: round(100 * s.mean())),
    mapillary_thay_pct=('mapillary_saw', lambda s: round(100 * s.mean()))).reset_index()
display(per_sign)
print('Trung vị box (px) khi thấy:', imgs.box_px.median())

In [ ]:
# 4. Xem mắt: 24 ảnh thấy (khung xanh) và 24 ảnh không thấy (vạch đỏ = hướng biển phải ở đó)
import matplotlib.pyplot as plt
from PIL import ImageDraw

def expected_x(row, w, h):
    if row.pano:
        return ((row.off_deg / 360 + 0.5) % 1) * w
    f = row.focal if row.focal == row.focal and row.focal else 0.8
    return w / 2 + f * max(w, h) * math.tan(math.radians(row.off_deg))

def grid(df, title, path):
    df = df.sample(min(24, len(df)), random_state=0)
    fig, axs = plt.subplots(4, 6, figsize=(24, 12))
    for ax in axs.flat: ax.axis('off')
    for ax, row in zip(axs.flat, df.itertuples()):
        im = Image.open(f'data/images/{row.image_id}.jpg').convert('RGB'); d = ImageDraw.Draw(im)
        x = expected_x(row, *im.size); d.line([(x, 0), (x, im.size[1])], fill='red', width=3)
        if row.conf == row.conf and row.conf >= 0.25: d.rectangle([row.x1, row.y1, row.x2, row.y2], outline='lime', width=4)
        ax.imshow(im); ax.set_title(f'{row.sign_type.split("--")[1]} · {row.dist_m:.0f} m · {row.captured[:4]}', fontsize=9)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=70); plt.show()

grid(imgs[imgs.conf >= 0.25], 'Thấy biển (conf ≥ 0.25)', 'seen.jpg')
grid(imgs[~(imgs.conf >= 0.25)], 'Không thấy', 'missed.jpg')

In [ ]:
# 5. Tải kết quả về
summary.to_csv('summary.csv', index=False, encoding='utf-8-sig')
per_sign.to_csv('per_sign.csv', index=False, encoding='utf-8-sig')
imgs.drop(columns=['dist_bin']).to_csv('per_image.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile('poc-results.zip', 'w') as z:
    for f in ['summary.csv', 'per_sign.csv', 'per_image.csv', 'seen.jpg', 'missed.jpg']: z.write(f)
files.download('poc-results.zip')